In [4]:
%load_ext autoreload
%autoreload 2
import sys, json, time
from pathlib import Path
from collections import Counter
import polars as pl

ROOT = Path.cwd()
while ROOT != ROOT.parent and not ((ROOT / 'src').is_dir() and (ROOT / 'configs').is_dir()):
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# import load_config giống notebook 04
from src.config import load_config
from src.features.prepare import META
cfg = load_config()
PROC = ROOT / 'data' / 'processed'

lf = pl.scan_parquet(PROC / 'model_table.parquet')
schema = lf.collect_schema()
FEATS = [c for c in schema.names() if c not in META]
print('Tổng cột:', len(schema), '| Feature:', len(FEATS))
print(Counter(str(schema[c]) for c in FEATS))

# Cỡ mẫu và default rate theo nhóm cửa sổ
win = (
    lf.filter(pl.col('split').is_in(['train', 'valid']))
    .with_columns(
        pl.when(pl.col('split') == 'valid').then(pl.lit('valid_44_51'))
        .when(pl.col('WEEK_NUM') >= 20).then(pl.lit('train_20_43'))
        .otherwise(pl.lit('train_0_19'))
        .alias('grp')
    )
    .group_by('grp')
    .agg(pl.len().alias('n'), pl.col('target').mean().alias('default_rate'))
    .sort('grp')
    .collect(engine='streaming')
)
print(win)

# Kiểm tra việc số 1: *_yrs_ago không còn giá trị âm
yrs = [c for c in FEATS if c.endswith('_yrs_ago')]
neg = (
    lf.select([(pl.col(c) < 0).sum().alias(c) for c in yrs])
    .collect(engine='streaming')
    .transpose(include_header=True, column_names=['n_neg'])
    .filter(pl.col('n_neg') > 0)
)
print('Số cột *_yrs_ago:', len(yrs))
print(neg)

Tổng cột: 649 | Feature: 643
Counter({'Float32': 480, 'UInt32': 114, 'String': 34, 'Int64': 9, 'Boolean': 6})
shape: (3, 3)
┌─────────────┬────────┬──────────────┐
│ grp         ┆ n      ┆ default_rate │
│ ---         ┆ ---    ┆ ---          │
│ str         ┆ u32    ┆ f64          │
╞═════════════╪════════╪══════════════╡
│ train_0_19  ┆ 318780 ┆ 0.028126     │
│ train_20_43 ┆ 522720 ┆ 0.030749     │
│ valid_44_51 ┆ 224421 ┆ 0.035157     │
└─────────────┴────────┴──────────────┘
Số cột *_yrs_ago: 12
shape: (0, 2)
┌────────┬───────┐
│ column ┆ n_neg │
│ ---    ┆ ---   │
│ str    ┆ u32   │
╞════════╪═══════╡
└────────┴───────┘


In [7]:
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

SEED = cfg['seed']
STR_COLS = [c for c in FEATS if schema[c] == pl.String]
CAT_IDX = [FEATS.index(c) for c in STR_COLS]
ES_WEEKS = (40, 43)                       # early stopping nội bộ, chung cho A và B
WINDOWS = {'A': (0, 39), 'B': (20, 39)}   # phần fit của từng cửa sổ

df = (
    lf.filter(pl.col('split').is_in(['train', 'valid']))
    .select(['WEEK_NUM', 'target', 'split'] + FEATS)
    .collect(engine='streaming')
)
print(df.shape, f'{df.estimated_size("gb"):.2f} GB', '| Số cột String:', len(STR_COLS))

def encode(fit_df, others):
    """Mã hóa cột String theo danh mục của fit_df; giá trị chưa gặp → null."""
    maps = {c: fit_df.get_column(c).drop_nulls().unique().sort().to_list() for c in STR_COLS}
    def _enc(d):
        return d.with_columns([
            pl.col(c).replace_strict(maps[c], list(range(len(maps[c]))),
                                     default=None, return_dtype=pl.Int32)
            for c in STR_COLS
        ])
    return _enc(fit_df), [_enc(d) for d in others]

def to_X(d):
    return d.select([pl.col(c).cast(pl.Float32) for c in FEATS]).to_numpy()

def gini(y, p):
    return 2 * roc_auc_score(y, p) - 1

def psi(expected, actual, n_bins=10):
    qs = np.unique(np.quantile(expected, np.linspace(0, 1, n_bins + 1)))
    qs[0], qs[-1] = -np.inf, np.inf
    e = np.clip(np.histogram(expected, qs)[0] / len(expected), 1e-6, None)
    a = np.clip(np.histogram(actual, qs)[0] / len(actual), 1e-6, None)
    return float(np.sum((a - e) * np.log(a / e)))

PARAMS = dict(
    objective='binary', metric='auc', learning_rate=0.05, num_leaves=63,
    min_child_samples=200, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
    lambda_l2=10.0, cat_smooth=10, min_data_per_group=200, max_cat_to_onehot=4,
    seed=SEED, deterministic=True, force_row_wise=True, verbose=-1, n_jobs=-1,
)
es_raw = df.filter(pl.col('WEEK_NUM').is_between(*ES_WEEKS))
va_raw = df.filter(pl.col('split') == 'valid')

(1065921, 646) 2.67 GB | Số cột String: 34


In [ ]:
results = {}

for name, (w0, w1) in WINDOWS.items():
    t0 = time.time()
    fit, (es, va) = encode(df.filter(pl.col('WEEK_NUM').is_between(w0, w1)), [es_raw, va_raw])
    X_fit, y_fit = to_X(fit), fit['target'].to_numpy()
    X_es, y_es = to_X(es), es['target'].to_numpy()
    X_va, y_va = to_X(va), va['target'].to_numpy()

    dtr = lgb.Dataset(X_fit, y_fit, feature_name=FEATS, categorical_feature=CAT_IDX)
    des = lgb.Dataset(X_es, y_es, reference=dtr)
    m = lgb.train(PARAMS, dtr, num_boost_round=3000, valid_sets=[des], valid_names=['es'],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])

    p_fit, p_es, p_va = (m.predict(X, num_iteration=m.best_iteration) for X in (X_fit, X_es, X_va))
    wk = va['WEEK_NUM'].to_numpy()
    g_wk = [gini(y_va[wk == w], p_va[wk == w]) for w in np.unique(wk)]

    results[name] = dict(
        weeks_fit=[w0, w1], n_fit=len(y_fit), best_iter=m.best_iteration,
        gini_fit=gini(y_fit, p_fit), gini_es=gini(y_es, p_es), gini_valid=gini(y_va, p_va),
        gap_fit_valid=gini(y_fit, p_fit) - gini(y_va, p_va),
        psi_score_valid=psi(p_fit, p_va),
        gini_valid_week_mean=float(np.mean(g_wk)), gini_valid_week_std=float(np.std(g_wk)),
        gini_valid_week_min=float(np.min(g_wk)), minutes=round((time.time() - t0) / 60, 1),
    )
    m.save_model(str(ROOT / 'reports' / f'lgb_window_{name}.txt'), num_iteration=m.best_iteration)
    print(name, json.dumps(results[name], indent=1))
    del X_fit, X_es, X_va, dtr, des, fit, es, va

(ROOT / 'reports' / 'train_window_ab.json').write_text(json.dumps(results, indent=1), encoding='utf-8')
print(pl.DataFrame([{'window': k, **v} for k, v in results.items()]).drop('weeks_fit'))

A {
 "weeks_fit": [
  0,
  39
 ],
 "n_fit": 754796,
 "best_iter": 145,
 "gini_fit": 0.8375147091246551,
 "gini_es": 0.6516107462036977,
 "gini_valid": 0.6550596194016984,
 "gap_fit_valid": 0.18245508972295665,
 "psi_score_valid": 0.0010062735596895684,
 "gini_valid_week_mean": 0.6524973917982382,
 "gini_valid_week_std": 0.02950925674379993,
 "gini_valid_week_min": 0.6189944648893577,
 "minutes": 3.2
}
B {
 "weeks_fit": [
  20,
  39
 ],
 "n_fit": 436016,
 "best_iter": 123,
 "gini_fit": 0.868749518905642,
 "gini_es": 0.6421757976987474,
 "gini_valid": 0.6446377523817062,
 "gap_fit_valid": 0.22411176652393583,
 "psi_score_valid": 0.0010422027728776984,
 "gini_valid_week_mean": 0.6420054494334531,
 "gini_valid_week_std": 0.026877175201699258,
 "gini_valid_week_min": 0.6126528373810269,
 "minutes": 1.8
}
shape: (2, 12)
┌────────┬────────┬───────────┬──────────┬───┬──────────────┬──────────────┬─────────────┬─────────┐
│ window ┆ n_fit  ┆ best_iter ┆ gini_fit ┆ … ┆ gini_valid_w ┆ gini_valid_

In [9]:
def run_window(w0, w1, seed):
    fit, (es, va) = encode(df.filter(pl.col('WEEK_NUM').is_between(w0, w1)), [es_raw, va_raw])
    X_fit, y_fit = to_X(fit), fit['target'].to_numpy()
    X_es, y_es = to_X(es), es['target'].to_numpy()
    X_va, y_va = to_X(va), va['target'].to_numpy()
    dtr = lgb.Dataset(X_fit, y_fit, feature_name=FEATS, categorical_feature=CAT_IDX)
    des = lgb.Dataset(X_es, y_es, reference=dtr)
    m = lgb.train({**PARAMS, 'seed': seed}, dtr, num_boost_round=3000,
                  valid_sets=[des], valid_names=['es'],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
    p_va = m.predict(X_va, num_iteration=m.best_iteration)
    wk = va['WEEK_NUM'].to_numpy()
    return gini(y_va, p_va), {int(w): gini(y_va[wk == w], p_va[wk == w]) for w in np.unique(wk)}

rows = []
for s in [SEED, SEED + 1, SEED + 2]:
    for name, (w0, w1) in WINDOWS.items():
        g, gw = run_window(w0, w1, s)
        rows.append({'seed': s, 'window': name, 'week': -1, 'gini': g})
        rows += [{'seed': s, 'window': name, 'week': w, 'gini': v} for w, v in gw.items()]
        print(s, name, round(g, 4))

res = pl.DataFrame(rows)
res.write_csv(ROOT / 'reports' / 'train_window_ab_seeds.csv')

print(res.filter(pl.col('week') == -1)
         .group_by('window')
         .agg(pl.col('gini').mean().alias('gini_valid_mean'),
              pl.col('gini').std(ddof=0).alias('gini_valid_std'))
         .sort('window'))

paired = (res.filter(pl.col('week') >= 0)
             .pivot(on='window', index=['seed', 'week'], values='gini')
             .with_columns((pl.col('A') - pl.col('B')).alias('diff')))
print(paired.group_by('week').agg(pl.col('diff').mean().alias('diff_mean')).sort('week'))
print('Số cặp (seed, tuần) có A > B:', int((paired['diff'] > 0).sum()), '/', paired.height)

42 A 0.6551
42 B 0.6446
43 A 0.6537
43 B 0.644
44 A 0.6559
44 B 0.6426
shape: (2, 3)
┌────────┬─────────────────┬────────────────┐
│ window ┆ gini_valid_mean ┆ gini_valid_std │
│ ---    ┆ ---             ┆ ---            │
│ str    ┆ f64             ┆ f64            │
╞════════╪═════════════════╪════════════════╡
│ A      ┆ 0.654899        ┆ 0.000904       │
│ B      ┆ 0.643735        ┆ 0.000862       │
└────────┴─────────────────┴────────────────┘
shape: (8, 2)
┌──────┬───────────┐
│ week ┆ diff_mean │
│ ---  ┆ ---       │
│ i64  ┆ f64       │
╞══════╪═══════════╡
│ 44   ┆ 0.015234  │
│ 45   ┆ 0.005641  │
│ 46   ┆ 0.008081  │
│ 47   ┆ 0.013797  │
│ 48   ┆ 0.015365  │
│ 49   ┆ 0.010564  │
│ 50   ┆ 0.011024  │
│ 51   ┆ 0.010235  │
└──────┴───────────┘
Số cặp (seed, tuần) có A > B: 24 / 24


In [10]:
TRAIN_WIN = {'A': (0, 43), 'B': (20, 43)}

def dist(s, edges=None):
    n = s.len()
    if edges is None:                                   # String, Boolean: mỗi giá trị một bin
        vc = s.cast(pl.String).fill_null('__NA__').value_counts()
        return dict(zip(vc[:, 0].to_list(), (vc['count'] / n).to_list()))
    x = s.drop_nulls().cast(pl.Float64).to_numpy()
    cnt = np.bincount(np.searchsorted(edges, x, side='right'), minlength=len(edges) + 1)
    d = {i: c / n for i, c in enumerate(cnt)}
    d['__NA__'] = s.null_count() / n                    # missing là một bin riêng
    return d

def psi_feat(e, a, n_bins=10):
    if e.dtype in (pl.String, pl.Boolean):
        de, da = dist(e), dist(a)
    else:
        x = e.drop_nulls().cast(pl.Float64).to_numpy()
        edges = (np.unique(np.quantile(x, np.linspace(0, 1, n_bins + 1)[1:-1]))
                 if len(x) else np.array([]))
        de, da = dist(e, edges), dist(a, edges)
    keys = list(set(de) | set(da))
    pe = np.clip([de.get(k, 0) for k in keys], 1e-6, None)
    pa = np.clip([da.get(k, 0) for k in keys], 1e-6, None)
    return float(np.sum((pa - pe) * np.log(pa / pe)))

rows = []
for name, (w0, w1) in TRAIN_WIN.items():
    tr = df.filter(pl.col('WEEK_NUM').is_between(w0, w1))
    rows += [{'feature': c, 'window': name, 'psi': psi_feat(tr[c], va_raw[c])} for c in FEATS]

psi_df = pl.DataFrame(rows).pivot(on='window', index='feature', values='psi')
psi_df.write_csv(ROOT / 'reports' / 'psi_valid_vs_window.csv')

for w in ['A', 'B']:
    print(w, '| > 0.25:', psi_df.filter(pl.col(w) > 0.25).height,
          '| 0.1–0.25:', psi_df.filter(pl.col(w).is_between(0.1, 0.25)).height,
          '| < 0.1:', psi_df.filter(pl.col(w) < 0.1).height)

print(psi_df.filter((pl.col('A') > 0.25) | (pl.col('B') > 0.25))
            .sort('A', descending=True).head(30))

A | > 0.25: 83 | 0.1–0.25: 25 | < 0.1: 535
B | > 0.25: 22 | 0.1–0.25: 6 | < 0.1: 615
shape: (30, 3)
┌─────────────────────────────────┬───────────┬──────────┐
│ feature                         ┆ A         ┆ B        │
│ ---                             ┆ ---       ┆ ---      │
│ str                             ┆ f64       ┆ f64      │
╞═════════════════════════════════╪═══════════╪══════════╡
│ requesttype_4525192L            ┆ 11.100389 ┆ 8.370071 │
│ credit_bureau_a_2__pmts_year_1… ┆ 6.067523  ┆ 8.40744  │
│ credit_bureau_a_2__pmts_year_1… ┆ 1.268555  ┆ 1.007921 │
│ credit_bureau_a_1__count        ┆ 1.103097  ┆ 0.228475 │
│ numinstpaid_4499208L            ┆ 0.987286  ┆ 0.492152 │
│ …                               ┆ …         ┆ …        │
│ credit_bureau_a_1__dpdmaxdatey… ┆ 0.55224   ┆ 0.036379 │
│ credit_bureau_a_1__lastupdate_… ┆ 0.551407  ┆ 0.038844 │
│ credit_bureau_a_1__dateofcrede… ┆ 0.550277  ┆ 0.029963 │
│ credit_bureau_a_1__dateofrealr… ┆ 0.549952  ┆ 0.033119 │
│ credit_bureau

In [11]:
pl.Config.set_fmt_str_lengths(80)
pl.Config.set_tbl_rows(40)

flag_B = psi_df.filter(pl.col('B') > 0.25)['feature'].to_list()
trB = df.filter(pl.col('WEEK_NUM').is_between(20, 43))

diag = (
    psi_df.filter(pl.col('B') > 0.25)
    .join(pl.DataFrame({
        'feature': flag_B,
        'dtype': [str(schema[c]) for c in flag_B],
        'na_trainB': trB.select([(pl.col(c).null_count() / pl.len()).alias(c) for c in flag_B]).row(0),
        'na_valid': va_raw.select([(pl.col(c).null_count() / pl.len()).alias(c) for c in flag_B]).row(0),
        'first_week': df.select([pl.col('WEEK_NUM').filter(pl.col(c).is_not_null()).min().alias(c)
                                 for c in flag_B]).row(0),
    }), on='feature')
    .sort('B', descending=True)
)
print(diag)

only_A = psi_df.filter((pl.col('A') > 0.25) & (pl.col('B') <= 0.25))
print('Chỉ lệch ở A:', only_A.height)
print(only_A.with_columns(
          pl.when(pl.col('feature').str.contains('__'))
            .then(pl.col('feature').str.split('__').list.first())
            .otherwise(pl.lit('depth0')).alias('src'))
      .group_by('src').len().sort('len', descending=True))

shape: (22, 7)
┌─────────────────────────────┬───────────┬──────────┬─────────┬───────────┬──────────┬────────────┐
│ feature                     ┆ A         ┆ B        ┆ dtype   ┆ na_trainB ┆ na_valid ┆ first_week │
│ ---                         ┆ ---       ┆ ---      ┆ ---     ┆ ---       ┆ ---      ┆ ---        │
│ str                         ┆ f64       ┆ f64      ┆ str     ┆ f64       ┆ f64      ┆ i64        │
╞═════════════════════════════╪═══════════╪══════════╪═════════╪═══════════╪══════════╪════════════╡
│ credit_bureau_a_2__pmts_yea ┆ 6.067523  ┆ 8.40744  ┆ Int64   ┆ 0.159806  ┆ 0.141983 ┆ 0          │
│ r_1139T__min_m              ┆           ┆          ┆         ┆           ┆          ┆            │
│ requesttype_4525192L        ┆ 11.100389 ┆ 8.370071 ┆ String  ┆ 0.59019   ┆ 0.0      ┆ 34         │
│ credit_bureau_a_2__pmts_yea ┆ 1.268555  ┆ 1.007921 ┆ UInt32  ┆ 0.159806  ┆ 0.141983 ┆ 0          │
│ r_1139T__n_months           ┆           ┆          ┆         ┆           ┆

In [12]:
chk = [
    'credit_bureau_a_2__pmts_year_1139T__min_m',
    'credit_bureau_a_2__pmts_year_1139T__n_months',
    'credit_bureau_a_1__dpdmaxdateyear_596T__mean_yrs_ago',
    'credit_bureau_a_1__overdueamountmaxdateyear_2T__mean_yrs_ago',
]
print(df.select(chk[:2]).describe())

by_wk = (df.group_by('WEEK_NUM').agg([pl.col(c).mean() for c in chk]).sort('WEEK_NUM'))
print('Spearman (trung bình theo tuần) với WEEK_NUM:')
print(by_wk.select([pl.corr('WEEK_NUM', c, method='spearman').alias(c) for c in chk]))
print(by_wk.gather_every(4))

shape: (9, 3)
┌────────────┬──────────────────────────────────────────┬──────────────────────────────────────────┐
│ statistic  ┆ credit_bureau_a_2__pmts_year_1139T__min_ ┆ credit_bureau_a_2__pmts_year_1139T__n_mo │
│ ---        ┆ m                                        ┆ nths                                     │
│ str        ┆ ---                                      ┆ ---                                      │
│            ┆ f64                                      ┆ f64                                      │
╞════════════╪══════════════════════════════════════════╪══════════════════════════════════════════╡
│ count      ┆ 869881.0                                 ┆ 869881.0                                 │
│ null_count ┆ 196040.0                                 ┆ 196040.0                                 │
│ mean       ┆ -22.39984                                ┆ 36.87462                                 │
│ std        ┆ 8.618704                                 ┆ 22.750325          

In [8]:
G2 = [c for c in FEATS if c.startswith('credit_bureau_a_2__pmts_year_1139T__')]
G3 = [c for c in FEATS if c.endswith('_yrs_ago')]
print('G2:', len(G2), G2)
print('G3:', len(G3))

# Thứ hạng của G2, G3 trong mô hình A đã lưu (theo gain)
mA = lgb.Booster(model_file=str(ROOT / 'reports' / 'lgb_window_A.txt'))
imp = (pl.DataFrame({'feature': mA.feature_name(), 'gain': mA.feature_importance('gain')})
         .sort('gain', descending=True).with_row_index('rank', offset=1))
print(imp.filter(pl.col('feature').is_in(G2 + G3)))

def run_A(feats, seed=SEED):
    cat_idx = [feats.index(c) for c in STR_COLS if c in feats]
    fit, (es, va) = encode(df.filter(pl.col('WEEK_NUM').is_between(0, 39)), [es_raw, va_raw])
    X = lambda d: d.select([pl.col(c).cast(pl.Float32) for c in feats]).to_numpy()
    dtr = lgb.Dataset(X(fit), fit['target'].to_numpy(), feature_name=feats, categorical_feature=cat_idx)
    des = lgb.Dataset(X(es), es['target'].to_numpy(), reference=dtr)
    m = lgb.train({**PARAMS, 'seed': seed}, dtr, num_boost_round=3000,
                  valid_sets=[des], valid_names=['es'],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
    p = m.predict(X(va), num_iteration=m.best_iteration)
    return gini(va['target'].to_numpy(), p), m.best_iteration

configs = {
    'full': FEATS,
    'drop_G2': [c for c in FEATS if c not in G2],
    'drop_G3': [c for c in FEATS if c not in G3],
    'drop_G2_G3': [c for c in FEATS if c not in G2 + G3],
}
abl = []
for k, feats in configs.items():
    g, it = run_A(feats)
    abl.append({'config': k, 'n_feat': len(feats), 'gini_valid': g, 'best_iter': it})
    print(k, len(feats), round(g, 4), it)

abl = pl.DataFrame(abl).with_columns(
    (pl.col('gini_valid') - pl.col('gini_valid').first()).alias('delta_vs_full'))
abl.write_csv(ROOT / 'reports' / 'ablation_time_features_A.csv')
print(abl)

G2: 3 ['credit_bureau_a_2__pmts_year_1139T__n_months', 'credit_bureau_a_2__pmts_year_1139T__n_contracts', 'credit_bureau_a_2__pmts_year_1139T__min_m']
G3: 12
shape: (15, 3)
┌──────┬─────────────────────────────────┬─────────────┐
│ rank ┆ feature                         ┆ gain        │
│ ---  ┆ ---                             ┆ ---         │
│ u32  ┆ str                             ┆ f64         │
╞══════╪═════════════════════════════════╪═════════════╡
│ 8    ┆ credit_bureau_a_1__dpdmaxdatey… ┆ 16208.40801 │
│ 74   ┆ credit_bureau_a_1__dpdmaxdatey… ┆ 960.445005  │
│ 76   ┆ credit_bureau_a_2__pmts_year_1… ┆ 948.435509  │
│ 94   ┆ credit_bureau_a_1__overdueamou… ┆ 606.49131   │
│ 124  ┆ credit_bureau_a_1__overdueamou… ┆ 369.100302  │
│ …    ┆ …                               ┆ …           │
│ 639  ┆ credit_bureau_a_1__overdueamou… ┆ 0.0         │
│ 640  ┆ credit_bureau_b_1__dpdmaxdatey… ┆ 0.0         │
│ 641  ┆ credit_bureau_b_1__dpdmaxdatey… ┆ 0.0         │
│ 642  ┆ credit_bureau_b_1__o

In [9]:
psi_saved = pl.read_csv(ROOT / 'reports' / 'psi_valid_vs_window.csv')
print(psi_saved.filter(pl.col('feature').is_in(G2)))
print(df.group_by('WEEK_NUM').agg(pl.col(G2).mean()).sort('WEEK_NUM')
        .select([pl.corr('WEEK_NUM', c, method='spearman').alias(c) for c in G2]))

shape: (3, 3)
┌─────────────────────────────────┬──────────┬──────────┐
│ feature                         ┆ A        ┆ B        │
│ ---                             ┆ ---      ┆ ---      │
│ str                             ┆ f64      ┆ f64      │
╞═════════════════════════════════╪══════════╪══════════╡
│ credit_bureau_a_2__pmts_year_1… ┆ 1.268555 ┆ 1.007921 │
│ credit_bureau_a_2__pmts_year_1… ┆ 0.028776 ┆ 0.009139 │
│ credit_bureau_a_2__pmts_year_1… ┆ 6.067523 ┆ 8.40744  │
└─────────────────────────────────┴──────────┴──────────┘
shape: (1, 3)
┌────────────────────────────────┬────────────────────────────────┬────────────────────────────────┐
│ credit_bureau_a_2__pmts_year_1 ┆ credit_bureau_a_2__pmts_year_1 ┆ credit_bureau_a_2__pmts_year_1 │
│ …                              ┆ …                              ┆ …                              │
│ ---                            ┆ ---                            ┆ ---                            │
│ f64                            ┆ f64        